# دانلود همهٔ ویدیوهای دوره سواد مالی

این نوت‌بوک ۴۱ فعالیت را از `course_page.html` می‌خواند، لینک پخش هر فعالیت را با نشست واردشدهٔ Chrome استخراج می‌کند و ویدیوها را در `data/raw_videos/` دانلود می‌کند.

> فقط محتوایی را دانلود کنید که مجاز به دسترسی و نگهداری آن هستید. فایل HTML، کوکی‌ها و ویدیوها نباید وارد Git شوند.

In [1]:
from pathlib import Path
from urllib.parse import urljoin
from http.cookiejar import MozillaCookieJar
import html
import json
import re
import shutil
import subprocess
import tempfile

import requests
from bs4 import BeautifulSoup

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DOWNLOAD_DIR = PROJECT_ROOT / "data" / "raw_videos"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

yt_dlp = shutil.which("yt-dlp")
if yt_dlp is None:
    local_binary = Path.home() / "bin" / "yt-dlp"
    if local_binary.exists():
        yt_dlp = str(local_binary)
    else:
        raise FileNotFoundError("yt-dlp پیدا نشد. ابتدا آن را نصب کنید: python3 -m pip install -U yt-dlp")

print("Project root:", PROJECT_ROOT)
print("Download folder:", DOWNLOAD_DIR)

Project root: /Users/macbookpro/Desktop/_PROGRAMING/0_transcription_rag_finance
Download folder: /Users/macbookpro/Desktop/_PROGRAMING/0_transcription_rag_finance/data/raw_videos


## ۱. خواندن همهٔ فعالیت‌ها

فایل `course_page.html` را در ریشهٔ پروژه یا کنار نوت‌بوک بگذارید.

In [2]:
BASE_URL = "https://lms.fintelligence.ir"
html_candidates = [CWD / "course_page.html", PROJECT_ROOT / "course_page.html"]
HTML_FILE = next((path for path in html_candidates if path.is_file()), None)

if HTML_FILE is None:
    checked = "\n".join(f"- {path}" for path in html_candidates)
    raise FileNotFoundError(f"course_page.html پیدا نشد. مسیرهای بررسی‌شده:\n{checked}")

with HTML_FILE.open("r", encoding="utf-8", errors="ignore") as file:
    soup = BeautifulSoup(file, "html.parser")

activities = []
seen_urls = set()
for link in soup.select("a.activity-ajax"):
    href = link.get("href")
    if not href or href == "#":
        continue
    url = urljoin(BASE_URL, href)
    if url in seen_urls:
        continue
    seen_urls.add(url)
    activities.append({
        "id": link.get("data-id"),
        "title": " ".join(link.get_text(" ", strip=True).split()),
        "url": url,
    })

if not activities:
    raise RuntimeError("هیچ فعالیتی در فایل HTML پیدا نشد.")

print("Activity count:", len(activities))
for index, activity in enumerate(activities, 1):
    print(f"{index:03d} | {activity['title']}")

Activity count: 41
001 | بیان اهمیت دورۀ در جستجوی خوشبختی 5 دقیقه
002 | در جستجوی خوشبختی از نگاه شرکت کنندگان کمپ دوم 3 دقیقه
003 | در جستجوی خوشبختی از نگاه شرکت کنندگان کمپ سوم 3 دقیقه
004 | در جستجوی خوشبختی از نگاه شرکت‌کنندگان کمپ پنجم 3 دقیقه
005 | الگوی تدریس دورۀ در جستجوی خوشبختی 6 دقیقه
006 | شیوۀ تدریس به بزرگسالان در دورۀ در جستجوی خوشبختی 7 دقیقه
007 | مخاطبان و پیش‌نیازهای شرکت در دورۀ در جستجوی خوشبختی 4 دقیقه
008 | وجه تسمیۀ نام دوره 3 دقیقه
009 | دوره‌ای برای یادگیری نه تدریس 6 دقیقه
010 | آوردۀ دوره برای شما 6 دقیقه
011 | 1.جایگاه سواد مالی در توسعۀ فردی؛ حرف حساب این همه حساب و کتاب 21 دقیقه
012 | 2.پلکان توانمندی مالی؛ شما روی کدام پله‌اید؟ 21 دقیقه
013 | 3.نگرانی مالی یا مسألۀ مالی؛ اندر حکایت پول 10 دقیقه
014 | 4.مدیریت هزینه؛ دخل نوزده، خرج بیست 14 دقیقه
015 | 5. الگوهای خرید؛ مردانه، زنانه، بچه‌گانه 18 دقیقه
016 | 6. صندوق‌های سه‌گانه؛ روز مبادا 11 دقیقه
017 | 7. بودجه‌بندی؛ کسری بودجه 12 دقیقه
018 | 8.بانک و سپرده‌گذاری؛ میز دریافت 15 دقیقه
019 | 9.قرض ناپسند

## ۲. استخراج امن لینک‌های پخش

این سلول نشست Chrome را با پاسخ واقعی ویدئو اعتبارسنجی می‌کند. لینک‌های تازه با `video_links.json` قبلی ادغام می‌شوند و در صورت قطع اینترنت یا نشست نامعتبر، manifest سالم قبلی بازنویسی نمی‌شود.


In [ ]:
COOKIE_TEST_URL = "https://lms.fintelligence.ir/video/play/farayad/62818/a6d37572c8d82cc7b4fca3914d9e39eb85295c96/master.m3u8"

def chrome_profile_specs():
    chrome_root = Path.home() / "Library" / "Application Support" / "Google" / "Chrome"
    specs = ["chrome"]
    if chrome_root.is_dir():
        profile_dirs = [
            path for path in chrome_root.iterdir()
            if path.is_dir() and (path.name == "Default" or path.name.startswith("Profile "))
        ]
        profile_dirs.sort(key=lambda path: (path.name != "Default", path.name))
        specs.extend(f"chrome:{path.name}" for path in profile_dirs)
    return list(dict.fromkeys(specs))

def export_browser_cookies(browser_spec, destination):
    destination.unlink(missing_ok=True)
    command = [
        yt_dlp,
        "--cookies-from-browser", browser_spec,
        "--cookies", str(destination),
        "--skip-download",
        "--no-warnings",
        COOKIE_TEST_URL,
    ]
    result = subprocess.run(command, check=False, capture_output=True, text=True)
    return destination.exists() and destination.stat().st_size > 0, result.stderr.strip()

def session_from_cookie_file(cookie_path):
    candidate_session = requests.Session()
    cookie_jar = MozillaCookieJar(str(cookie_path))
    cookie_jar.load(ignore_discard=True, ignore_expires=True)
    candidate_session.cookies = cookie_jar
    candidate_session.headers.update({
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/140.0.0.0 Safari/537.36",
        "X-Requested-With": "XMLHttpRequest",
        "Accept": "application/json, text/javascript, */*; q=0.01",
    })
    return candidate_session

def session_is_logged_in(candidate_session):
    """Only accept a session that returns an actual playable activity."""
    try:
        response = candidate_session.get(
            activities[0]["url"],
            params={"ajax": 1},
            headers={"Referer": f"{BASE_URL}/course/view/12518"},
            timeout=30,
            allow_redirects=True,
        )
        response.raise_for_status()
        payload_text = json.dumps(response.json(), ensure_ascii=False)
    except (requests.RequestException, ValueError):
        return False

    blocked = "/login" in response.url or "/user/session" in response.url
    has_video = ".m3u8" in payload_text.lower() or ".mp4" in payload_text.lower()
    return not blocked and has_video

cookie_file = Path(tempfile.gettempdir()) / "fintelligence_cookies.txt"
session = None
selected_browser_profile = None
cookie_errors = []

for browser_spec in chrome_profile_specs():
    exported, export_error = export_browser_cookies(browser_spec, cookie_file)
    if not exported:
        cookie_errors.append(f"{browser_spec}: {export_error or 'cookie export failed'}")
        continue
    candidate_session = session_from_cookie_file(cookie_file)
    if session_is_logged_in(candidate_session):
        session = candidate_session
        selected_browser_profile = browser_spec
        break

if session is None:
    details = "\n".join(cookie_errors[-3:])
    raise RuntimeError(
        "هیچ پروفایل واردشدهٔ Chrome برای LMS پیدا نشد. "
        "دوره را در Chrome باز کنید، مطمئن شوید ویدئو پخش می‌شود و Chrome را کامل ببندید؛ "
        "سپس این سلول را دوباره اجرا کنید."
        + (f"\nجزئیات:\n{details}" if details else "")
    )

cookie_available = True
print("Chrome profile:", selected_browser_profile)
print("نشست LMS معتبر است؛ استخراج همهٔ فعالیت‌ها شروع می‌شود.")

STREAM_PATTERN = re.compile(
    r"""(?P<url>(?:https?:)?//[^\s"'<>\\]+?\.(?:m3u8|mp4)(?:\?[^\s"'<>\\]*)?|/[^\s"'<>\\]+?\.(?:m3u8|mp4)(?:\?[^\s"'<>\\]*)?)""",
    re.IGNORECASE,
)
ROUTE_PATTERN = re.compile(
    r"""(?P<url>/(?:video|media|stream|player|file)/(?![^"'<>]*\.(?:js|css|png|jpe?g|gif|svg|woff2?))[^"'<>\\\s]+)""",
    re.IGNORECASE,
)
URL_PATTERN = re.compile(r"""https?://[^\s"'<>\\]+""", re.IGNORECASE)
ASSET_EXTENSIONS = (".js", ".css", ".png", ".jpg", ".jpeg", ".gif", ".svg", ".ico", ".woff", ".woff2")

def normalize_payload_text(value):
    if isinstance(value, (dict, list)):
        value = json.dumps(value, ensure_ascii=False)
    text = html.unescape(str(value))
    for _ in range(3):
        text = (
            text.replace("\\/", "/")
                .replace("\\u0026", "&")
                .replace("\\u003d", "=")
                .replace("\\u002F", "/")
                .replace("&amp;", "&")
        )
    return text

def stream_candidates(value, base_url):
    text = normalize_payload_text(value)
    found = []
    for match in STREAM_PATTERN.finditer(text):
        candidate = match.group("url").rstrip("),;]")
        if candidate.startswith("//"):
            candidate = "https:" + candidate
        found.append(urljoin(base_url, candidate))
    return list(dict.fromkeys(found))

def follow_candidates(value, base_url):
    text = normalize_payload_text(value)
    soup_fragment = BeautifulSoup(text, "html.parser")
    found = []
    for tag in soup_fragment.find_all(True):
        for attribute in ("src", "href", "data-src", "data-url", "data-file"):
            candidate = tag.get(attribute)
            if candidate:
                found.append(urljoin(base_url, candidate))
    found += [urljoin(base_url, m.group("url")) for m in ROUTE_PATTERN.finditer(text)]
    found += [m.group(0).rstrip("),;]") for m in URL_PATTERN.finditer(text)]
    cleaned = []
    for candidate in found:
        lowered = candidate.lower().split("?", 1)[0]
        if lowered.endswith(ASSET_EXTENSIONS):
            continue
        if candidate.startswith(BASE_URL) or any(word in lowered for word in ("player", "video", "media", "stream", "embed", "file")):
            cleaned.append(candidate)
    return list(dict.fromkeys(cleaned))

def resolve_stream(activity, max_pages=15):
    queue = [(activity["url"], {"ajax": 1}, f"{BASE_URL}/course/view/12518")]
    visited = set()
    debug_pages = []

    while queue and len(visited) < max_pages:
        page_url, params, referer = queue.pop(0)
        key = (page_url, tuple(sorted((params or {}).items())))
        if key in visited:
            continue
        visited.add(key)

        response = session.get(
            page_url,
            params=params,
            headers={"Referer": referer},
            timeout=45,
            allow_redirects=True,
        )
        try:
            payload = response.json()
        except ValueError:
            payload = response.text

        debug_pages.append({
            "input_url": page_url,
            "params": params,
            "referer": referer,
            "status_code": response.status_code,
            "final_url": response.url,
            "redirect_chain": [
                {"status_code": item.status_code, "url": item.url, "location": item.headers.get("location")}
                for item in response.history
            ],
            "content_type": response.headers.get("content-type"),
            "response_headers": dict(response.headers),
            "body": normalize_payload_text(payload)[:200000],
        })

        auth_redirect = "/login" in response.url or "/user/session" in response.url
        if auth_redirect:
            continue
        response.raise_for_status()

        streams = stream_candidates(payload, response.url)
        if streams:
            preferred = next((u for u in streams if ".m3u8" in u.lower()), streams[0])
            return preferred, debug_pages

        for candidate in follow_candidates(payload, response.url):
            candidate_path = candidate.lower().split("?", 1)[0]
            if any(blocked in candidate_path for blocked in ("/login", "/logout", "/user/session")):
                continue
            if candidate != response.url:
                queue.append((candidate, None, response.url))

    return None, debug_pages

links_file = DOWNLOAD_DIR / "video_links.json"
existing_videos = []
if links_file.is_file() and links_file.stat().st_size > 2:
    try:
        loaded_links = json.loads(links_file.read_text(encoding="utf-8"))
        if isinstance(loaded_links, list):
            existing_videos = [
                item for item in loaded_links
                if isinstance(item, dict) and item.get("name") and item.get("stream_url")
            ]
    except (OSError, json.JSONDecodeError):
        print("هشدار: manifest قبلی قابل خواندن نیست؛ فایل جدید فقط پس از استخراج موفق نوشته می‌شود.")

videos_by_name = {item["name"]: item for item in existing_videos}
fresh_links_found = 0
failed_activities = []
debug_dir = DOWNLOAD_DIR / "_debug"
debug_dir.mkdir(parents=True, exist_ok=True)

for index, activity in enumerate(activities, 1):
    debug_pages = []
    try:
        stream_url, debug_pages = resolve_stream(activity)
    except Exception as error:
        stream_url = None
        debug_pages.append({
            "activity": activity,
            "error_type": type(error).__name__,
            "error": str(error),
        })

    if stream_url is None:
        debug_file = debug_dir / f"activity-{index:03d}.json"
        debug_file.write_text(
            json.dumps(debug_pages, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        failed_activities.append({**activity, "debug_file": str(debug_file)})
        print(f"{index:03d} | NOT FOUND | {activity['title']}")
        continue

    video = {
        **activity,
        "name": f"session-{index:02d}",
        "stream_url": stream_url,
    }
    videos_by_name[video["name"]] = video
    fresh_links_found += 1
    print(f"{index:03d} | FOUND | {activity['title']}")

videos = sorted(
    videos_by_name.values(),
    key=lambda item: int(item["name"].split("-")[-1]),
)
print(f"\nFresh links found: {fresh_links_found} / {len(activities)}")
print(f"Usable links after merge: {len(videos)} / {len(activities)}")
if failed_activities:
    print("موارد ناموفق و فایل عیب‌یابی:")
    for item in failed_activities:
        print("-", item["debug_file"], "|", item["url"])

if fresh_links_found == 0:
    raise RuntimeError(
        "هیچ لینک تازه‌ای دریافت نشد؛ manifest قبلی دست‌نخورده ماند. "
        "در Chrome دوباره وارد LMS شوید، پخش یک ویدئو را بررسی کنید و سلول را دوباره اجرا کنید."
    )

manifest_tmp = links_file.with_suffix(".json.tmp")
manifest_tmp.write_text(json.dumps(videos, ensure_ascii=False, indent=2), encoding="utf-8")
manifest_tmp.replace(links_file)
print("Local link manifest:", links_file)


## ۳. دانلود همهٔ ویدیوها

این سلول تمام لینک‌های پیدا‌شده را دانلود می‌کند. فایل MP4 موجود و غیرخالی دوباره دانلود نمی‌شود. خطای یک قسمت مانع ادامهٔ بقیه نمی‌شود.

In [ ]:
links_file = DOWNLOAD_DIR / "video_links.json"
if "videos" not in globals() or not videos:
    if not links_file.is_file():
        raise RuntimeError("فایل video_links.json پیدا نشد؛ ابتدا سلول استخراج را اجرا کنید.")
    videos = json.loads(links_file.read_text(encoding="utf-8"))

if not videos:
    raise RuntimeError(
        "هیچ لینک قابل استفاده‌ای در video_links.json نیست؛ "
        "پس از ورود دوباره به LMS، سلول استخراج را اجرا کنید."
    )

download_errors = []
for index, video in enumerate(videos, 1):
    completed_file = DOWNLOAD_DIR / f"{video['name']}.mp4"
    if completed_file.is_file() and completed_file.stat().st_size > 0:
        print(f"SKIP {video['name']}: already downloaded")
        continue

    output_template = str(DOWNLOAD_DIR / f"{video['name']}.%(ext)s")
    command = [
        yt_dlp,
        "--referer", video["url"],
        "--merge-output-format", "mp4",
        "--continue",
        "--no-overwrites",
        "--socket-timeout", "30",
        "--retries", "3",
        "--fragment-retries", "3",
        "--retry-sleep", "fragment:exp=1:20",
        "-o", output_template,
        video["stream_url"],
    ]
    if cookie_available:
        command[1:1] = ["--cookies", str(cookie_file)]
    print(f"\n[{index}/{len(videos)}] Downloading {video['name']} | {video['title']}")
    result = subprocess.run(command, check=False)
    if result.returncode != 0:
        download_errors.append(video)
        print(f"FAILED: {video['name']} (continuing)")

downloaded_files = sorted(DOWNLOAD_DIR.glob("session-*.mp4"))
print(f"\nDownloaded files: {len(downloaded_files)}")
print(f"Download errors: {len(download_errors)}")
for item in download_errors:
    print("-", item["name"], item["url"])


## ۴. بررسی نتیجه

In [ ]:
downloaded_files = sorted(DOWNLOAD_DIR.glob("session-*.mp4"))
expected_names = {video["name"] for video in videos}
downloaded_names = {path.stem for path in downloaded_files}
missing_names = sorted(expected_names - downloaded_names)

print("Activities:", len(activities))
print("Video links:", len(videos))
print("Downloaded:", len(downloaded_files))
print("Missing:", len(missing_names))
if missing_names:
    print("Missing sessions:", ", ".join(missing_names))

ffprobe = shutil.which("ffprobe")
if ffprobe is None:
    print("ffprobe نصب نیست؛ برای بررسی فنی در macOS اجرا کنید: brew install ffmpeg")
else:
    invalid_files = []
    for file_path in downloaded_files:
        result = subprocess.run(
            [ffprobe, "-v", "error", "-select_streams", "v:0", "-show_entries", "stream=codec_name", "-of", "csv=p=0", str(file_path)],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0 or not result.stdout.strip():
            invalid_files.append(file_path.name)
    print("Invalid files:", len(invalid_files))
    for name in invalid_files:
        print("-", name)